# 🇰🇪 Fine-Tuning Meta NLLB-200 for English ➡️ Ekegusii (Gusii) NMT
This notebook contains self-contained code chunks to train a Meta NLLB-200 model on English to Ekegusii parallel sentences.

## 🛠️ Step 1: Environment Setup & Clone GitHub Repository

In [ ]:
# Clone repository and install dependencies
!git clone https://github.com/aykahsay/Multilogual_transaltion_nlp.git
%cd Multilogual_transaltion_nlp
!pip install -q transformers datasets evaluate sacrebleu sentencepiece sacremoses torch accelerate pandas scikit-learn tqdm

import torch
print("GPU Available:", torch.cuda.is_available())

## 📊 Step 2: Load Ekegusii Parallel Dataset

In [ ]:
import os
import pandas as pd

data_path = "data/languages/PSA_English_Ekegusii.csv"
df = pd.read_csv(data_path, dtype=str).dropna(subset=["English", "Ekegusii"])
print(f"Loaded {len(df):,} English-Ekegusii parallel pairs from {data_path}")
print(df.head(2).to_dict(orient="records"))

## 🚀 Step 3: Model Tokenization & Training Setup

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer
from datasets import Dataset

model_checkpoint = "facebook/nllb-200-distilled-600M"
output_dir = "models/nllb-en-guz"
os.makedirs(output_dir, exist_ok=True)

print("Loading pretrained model & tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, src_lang="eng_Latn", tgt_lang="guz_Latn")
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

# 90% train / 10% validation split
shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)
split_idx = int(0.9 * len(shuffled))
train_dataset = Dataset.from_pandas(shuffled.iloc[:split_idx])
val_dataset = Dataset.from_pandas(shuffled.iloc[split_idx:])

def preprocess(examples):
    inputs = [str(x) for x in examples["English"]]
    targets = [str(x) for x in examples["Ekegusii"]]
    model_inputs = tokenizer(inputs, max_length=128, truncation=True)
    labels = tokenizer(text_target=targets, max_length=128, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_dataset.map(preprocess, batched=True, remove_columns=train_dataset.column_names)
tokenized_val = val_dataset.map(preprocess, batched=True, remove_columns=val_dataset.column_names)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    save_strategy="epoch",
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator,
)

print("Starting Ekegusii Training...")
trainer.train()
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Saved to {output_dir}")